In [ ]:
pip install requests xarray netCDF4 tqdm

In [ ]:
import requests
import xarray as xr
import numpy as np
from pathlib import Path
from tqdm import tqdm
import time

# Brazil bounding box
BBOX = {
    'min_lon': -94.1875,
    'max_lon': 37.0625,
    'min_lat': -39.0208,
    'max_lat': 18.2292
}

# NASA POWER parameters (commonly used meteorological variables)
PARAMETERS = [
    'T2M',              # Temperature at 2 Meters (°C)
    'T2M_MAX',          # Max Temperature at 2 Meters (°C)
    'T2M_MIN',          # Min Temperature at 2 Meters (°C)
    'PRECTOTCORR',      # Precipitation Corrected (mm/day)
    'RH2M',             # Relative Humidity at 2 Meters (%)
    'WS2M',             # Wind Speed at 2 Meters (m/s)
    'ALLSKY_SFC_SW_DWN', # All Sky Surface Shortwave Downward Irradiance (kW-hr/m^2/day)
    'ALLSKY_SFC_LW_DWN', # All Sky Surface Longwave Downward Irradiance (kW-hr/m^2/day)
    'PS',               # Surface Pressure (kPa)
]

# Output directory (Google Drive)
OUTPUT_DIR = Path(r"Q:\My Drive\Brazil_NASA_POWER_Daily")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Date range
START_DATE = '20250101'
END_DATE = '20251231'

print("=" * 70)
print("NASA POWER Download Configuration")
print("=" * 70)
print(f"Bounding Box: {BBOX}")
print(f"Date Range: {START_DATE} to {END_DATE}")
print(f"Parameters: {len(PARAMETERS)}")
print(f"Output: {OUTPUT_DIR}")
print("=" * 70)

In [ ]:
# Function to download NASA POWER data using regional API
def download_nasa_power_region(bbox, parameters, start_date, end_date, output_file):
    """
    Download NASA POWER data for a bounding box region.
    Uses the regional API endpoint which returns NetCDF format.
    """
    
    # Build API request URL
    base_url = "https://power.larc.nasa.gov/api/temporal/daily/regional"
    
    params = {
        'parameters': ','.join(parameters),
        'community': 'AG',  # Agricultural community
        'longitude-min': bbox['min_lon'],
        'longitude-max': bbox['max_lon'],
        'latitude-min': bbox['min_lat'],
        'latitude-max': bbox['max_lat'],
        'start': start_date,
        'end': end_date,
        'format': 'NETCDF',
        'time-standard': 'UTC'
    }
    
    try:
        print(f"\nDownloading: {output_file.name}")
        print(f"Requesting data from NASA POWER API...")
        
        # Make request with extended timeout
        response = requests.get(base_url, params=params, timeout=600, stream=True)
        
        if response.status_code == 200:
            # Get total size for progress bar
            total_size = int(response.headers.get('content-length', 0))
            
            # Download with progress bar
            with open(output_file, 'wb') as f:
                with tqdm(total=total_size, unit='B', unit_scale=True) as pbar:
                    for chunk in response.iter_content(chunk_size=8192):
                        f.write(chunk)
                        pbar.update(len(chunk))
            
            print(f"✓ Successfully downloaded: {output_file}")
            print(f"  File size: {output_file.stat().st_size / 1024 / 1024:.2f} MB")
            return True
            
        else:
            print(f"✗ HTTP Error {response.status_code}")
            if response.status_code == 400:
                print(f"  Error: {response.text[:500]}")
                print(f"  Note: Data may not be available for requested date range")
            return False
            
    except requests.exceptions.Timeout:
        print(f"✗ Request timeout - NASA POWER API may be slow or unavailable")
        return False
    except Exception as e:
        print(f"✗ Error: {str(e)}")
        return False

In [ ]:
# Download full year data as single NetCDF
output_file = OUTPUT_DIR / f"NASA_POWER_Brazil_Daily_{START_DATE}_{END_DATE}.nc"

if output_file.exists():
    print(f"\n⚠️  File already exists: {output_file}")
    print("   Delete it first if you want to re-download")
else:
    success = download_nasa_power_region(
        bbox=BBOX,
        parameters=PARAMETERS,
        start_date=START_DATE,
        end_date=END_DATE,
        output_file=output_file
    )
    
    if success:
        print("\n" + "=" * 70)
        print("✓ NASA POWER Download Complete!")
        print("=" * 70)
        print(f"File: {output_file}")
        print("\nNext steps:")
        print("1. Convert NetCDF to GeoTIFF rasters using processing scripts")
        print("2. Resample to 0.02° resolution")
        print("3. Upload to covariables2 database")
        print("=" * 70)
    else:
        print("\n" + "=" * 70)
        print("✗ Download Failed")
        print("=" * 70)
        print("Possible reasons:")
        print("• 2025 data not yet available (typical 2-6 month delay)")
        print("• NASA POWER API temporarily unavailable")
        print("• Network timeout or connection issue")
        print("\nTry again later or check: https://power.larc.nasa.gov/")
        print("=" * 70)

In [ ]:
# Optional: Quick inspection of downloaded NetCDF
if output_file.exists():
    print("\n" + "=" * 70)
    print("NetCDF File Inspection")
    print("=" * 70)
    
    try:
        ds = xr.open_dataset(output_file)
        
        print(f"\nDimensions: {dict(ds.dims)}")
        print(f"\nVariables: {list(ds.data_vars)}")
        print(f"\nDate range: {ds.time.values[0]} to {ds.time.values[-1]}")
        print(f"Total days: {len(ds.time)}")
        
        print("\n" + "=" * 70)
        ds.close()
        
    except Exception as e:
        print(f"Error reading NetCDF: {e}")
else:
    print("\nNo file to inspect. Download first.")

## Alternative: Download by Month

If downloading the full year fails (file too large), uncomment below to download month by month:

In [ ]:
# # Download month by month for 2025
# months = [
#     ('20250101', '20250131', 'Jan'),
#     ('20250201', '20250228', 'Feb'),
#     ('20250301', '20250331', 'Mar'),
#     ('20250401', '20250430', 'Apr'),
#     ('20250501', '20250531', 'May'),
#     ('20250601', '20250630', 'Jun'),
#     ('20250701', '20250731', 'Jul'),
#     ('20250801', '20250831', 'Aug'),
#     ('20250901', '20250930', 'Sep'),
#     ('20251001', '20251031', 'Oct'),
#     ('20251101', '20251130', 'Nov'),
#     ('20251201', '20251231', 'Dec'),
# ]

# success_count = 0
# failed_count = 0

# for start, end, month_name in months:
#     output_file = OUTPUT_DIR / f"NASA_POWER_Brazil_Daily_2025_{month_name}.nc"
    
#     if output_file.exists():
#         print(f"\n⏭️  Skipping {month_name} (already exists)")
#         success_count += 1
#         continue
    
#     success = download_nasa_power_region(
#         bbox=BBOX,
#         parameters=PARAMETERS,
#         start_date=start,
#         end_date=end,
#         output_file=output_file
#     )
    
#     if success:
#         success_count += 1
#     else:
#         failed_count += 1
#         print(f"\n⚠️  Stopping at {month_name} - data likely not available yet")
#         break
    
#     # Rate limiting
#     time.sleep(2)

# print("\n" + "=" * 70)
# print(f"Monthly Download Summary: {success_count} success, {failed_count} failed")
# print("=" * 70)